In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time


## Data Reading

In [0]:
df = spark.read.format("parquet")\
          .load("abfss://bronze@databrickssuraj.dfs.core.windows.net/products") 

In [0]:
df.display()

product_id,product_name,category,brand,price,_rescued_data
P0001,Clearly Its,Beauty,Nike,1868.54,null
P0002,Production Clear,Beauty,Apple,587.13,null
P0003,Culture Coach,Home,Revlon,1599.24,null
P0004,Movement Part,Sports,LG,651.71,null
P0005,Fact Name,Clothing,Samsung,1861.78,null
P0006,Usually Stop,Toys,Adidas,936.36,null
P0007,Reveal Current,Sports,Adidas,1954.02,null
P0008,Force Language,Beauty,Puma,1251.26,null
P0009,Stage Leg,Clothing,Samsung,1247.15,null
P0010,Leader Then,Sports,Sony,975.53,null


In [0]:
df = df.drop("_rescued_data")
df.display()

product_id,product_name,category,brand,price
P0001,Clearly Its,Beauty,Nike,1868.54
P0002,Production Clear,Beauty,Apple,587.13
P0003,Culture Coach,Home,Revlon,1599.24
P0004,Movement Part,Sports,LG,651.71
P0005,Fact Name,Clothing,Samsung,1861.78
P0006,Usually Stop,Toys,Adidas,936.36
P0007,Reveal Current,Sports,Adidas,1954.02
P0008,Force Language,Beauty,Puma,1251.26
P0009,Stage Leg,Clothing,Samsung,1247.15
P0010,Leader Then,Sports,Sony,975.53


### **Functions**

In [0]:
df.createOrReplaceTempView("products")


In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cata.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
RETURN p_price * 0.90

In [0]:
%sql
select product_id,price,databricks_cata.bronze.discount_func(price) as discounted_price
from products

product_id,price,discounted_price
P0001,1868.54,1681.686
P0002,587.13,528.417
P0003,1599.24,1439.316
P0004,651.71,586.5390000000001
P0005,1861.78,1675.602
P0006,936.36,842.724
P0007,1954.02,1758.618
P0008,1251.26,1126.134
P0009,1247.15,1122.4350000000002
P0010,975.53,877.977


In [0]:
df = df.withColumn("discounted_price",expr("databricks_cata.bronze.discount_func(price)"))
df.display()

product_id,product_name,category,brand,price,discounted_price
P0001,Clearly Its,Beauty,Nike,1868.54,1681.686
P0002,Production Clear,Beauty,Apple,587.13,528.417
P0003,Culture Coach,Home,Revlon,1599.24,1439.316
P0004,Movement Part,Sports,LG,651.71,586.5390000000001
P0005,Fact Name,Clothing,Samsung,1861.78,1675.602
P0006,Usually Stop,Toys,Adidas,936.36,842.724
P0007,Reveal Current,Sports,Adidas,1954.02,1758.618
P0008,Force Language,Beauty,Puma,1251.26,1126.134
P0009,Stage Leg,Clothing,Samsung,1247.15,1122.4350000000002
P0010,Leader Then,Sports,Sony,975.53,877.977


In [0]:
%sql
CREATE OR REPLACEN FUNCTION databricks_cata.bronze.upper_func(p_brand STRING)
RETURN STRING
LANGUAGE PYTHON
AS
$$
   for i p_brand:
      if i.isupper():
          return p_brand 
$$

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cata.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
   return p_brand.upper()
$$

In [0]:
%sql
select product_id,brand ,databricks_cata.bronze.upper_func(brand) as brand_upper
from products

product_id,brand,brand_upper
P0001,Nike,NIKE
P0002,Apple,APPLE
P0003,Revlon,REVLON
P0004,LG,LG
P0005,Samsung,SAMSUNG
P0006,Adidas,ADIDAS
P0007,Adidas,ADIDAS
P0008,Puma,PUMA
P0009,Samsung,SAMSUNG
P0010,Sony,SONY


In [0]:
df.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://silver@databrickssuraj.dfs.core.windows.net/products")\
        .save()    

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_cata.silver.products_silver
USING DELTA
LOCATION "abfss://silver@databrickssuraj.dfs.core.windows.net/products"

In [0]:
%sql
select * from databricks_cata.silver.products_silver

product_id,product_name,category,brand,price,discounted_price
P0001,Clearly Its,Beauty,Nike,1868.54,1681.686
P0002,Production Clear,Beauty,Apple,587.13,528.417
P0003,Culture Coach,Home,Revlon,1599.24,1439.316
P0004,Movement Part,Sports,LG,651.71,586.5390000000001
P0005,Fact Name,Clothing,Samsung,1861.78,1675.602
P0006,Usually Stop,Toys,Adidas,936.36,842.724
P0007,Reveal Current,Sports,Adidas,1954.02,1758.618
P0008,Force Language,Beauty,Puma,1251.26,1126.134
P0009,Stage Leg,Clothing,Samsung,1247.15,1122.4350000000002
P0010,Leader Then,Sports,Sony,975.53,877.977
